In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import shutil
import yaml
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
DRIVE_DIR = "/content/drive/MyDrive/ParkFlow_AI"
DRIVE_ZIP_PATH = os.path.join(DRIVE_DIR, "top_view_dataset_processed.zip")

LOCAL_RAW = Path("/content/raw_data")
LOCAL_TOP = Path("/content/top_view_dataset")

os.makedirs(DRIVE_DIR, exist_ok=True)

# 3. Kaggle Credentials (Replace with your actual keys)
os.environ['KAGGLE_USERNAME'] = "sandakannipunajith"
os.environ['KAGGLE_KEY'] = "KGAT_8eb82a391617f56c3c05696cdb9f87fc"

Mounted at /content/drive


In [ ]:
# if os.path.exists(DRIVE_ZIP_PATH):
#     print("🚀 Restoring processed dataset from Drive...")
#     shutil.copy2(DRIVE_ZIP_PATH, "/content/top_view_dataset.zip")
#     !unzip -q /content/top_view_dataset.zip -d /content/
# else:
#     print("📂 Starting raw conversion...")
#     !kaggle datasets download -d farzadnekouei/top-view-vehicle-detection-image-dataset -p {LOCAL_RAW}
#     !unzip -q {LOCAL_RAW}/top-view-vehicle-detection-image-dataset.zip -d {LOCAL_RAW}

    # As per your dataset structure: 0:spaces, 1:space-empty, 2:space-occupied
    # category_map = {0: 0, 1: 1, 2: 2}

    # # Generate data.yaml with your exact 3 classes
    # data_yaml = {
    #     'path': str(LOCAL_TOP),
    #     'train': 'train/images',
    #     'val': 'valid/images',
    #     'test': 'test/images',
    #     'nc': 1,
    #     'names': ['Vehicle']
    # }
    # with open(LOCAL_YOLO / "data.yaml", "w") as f:
    #     yaml.dump(data_yaml, f)

    # Backup to Drive
shutil.copy2("/content/raw_data/top-view-vehicle-detection-image-dataset.zip", DRIVE_ZIP_PATH)
print("✅ Conversion and Drive backup complete.")

✅ Conversion and Drive backup complete.


In [ ]:
from ultralytics import YOLO

# 1. Load your Phase 1 weights (PKLot trained)
# This model already knows what "car roofs" look like in general
model = YOLO('/content/drive/MyDrive/ParkFlow_AI/new_best.pt')

# 2. Fine-tune on the specialized Top-View dataset
model.train(
    data='/content/raw_data/Vehicle_Detection_Image_Dataset/data.yaml',
    epochs=50,               # Fewer epochs needed for fine-tuning
    imgsz=640,
    batch=16,                # Adjust based on Colab T4 memory
    lr0=0.001,               # Lower learning rate to preserve PKLot knowledge
    degrees=180.0,           # CRITICAL: Allow full rotation for top-down view
    flipud=0.5,              # Vertical flips are realistic from above
    mosaic=1.0,              # Helps with small vehicles
    name='parkflow_final_vehicle_model',   # Sub-folder for this specific run
    project=f"{DRIVE_DIR}/training_runs",
    exist_ok=True            # Overwrite if you restart the cell
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/raw_data/Vehicle_Detection_Image_Dataset/data.yaml, degrees=180.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/ParkFlow_AI/new_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parkflow_final_vehicle_model, nbs=64, nms=Fals

In [ ]:

# 1. Path to your best weights in Google Drive
best_model_path = "/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.pt"

# 2. Load the trained model
model = YOLO(best_model_path)

# 3. Export the model to multiple formats
# ONNX is the most universal for deployment
print("Exporting to ONNX...")
model.export(format='onnx', dynamic=True)

# OpenVINO is great if you ever move to Intel CPUs
# CoreML is best for your MacBook Air M4
print("Exporting to CoreML...")
model.export(format='coreml')

print("All exports complete. Files are located in the same folder as your best.pt")

Exporting to ONNX...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 298ms
Prepared 4 packages in 7.62s
Installed 4 packages in 267ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime-gpu==1.24.2
 + onnxslim==0.1.86

requirements: AutoUpdate success ✅ 8.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to t

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 22 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 13.0s, saved as '/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.onnx' (36.9 MB)

Export complete (14.0s)
Results saved to /content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights
Predict:         yolo predict task=detect model=/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.onnx imgsz=640 data=/content/raw_data/Vehicle_Detection_Image_Dataset/data.yaml  
Visualize:       https://netron.app
Exporting to CoreML...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_fi

/usr/local/lib/python3.12/dist-packages/coremltools/optimize/torch/palettization/fake_palettize.py:82: SyntaxWarning: invalid escape sequence '\_'
  n_bits (:obj:`int`): Number of palettization bits. There would be :math:`2^{n\_bits}` unique weights in the ``LUT``.



CoreML: starting export with coremltools 9.0...


/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py:178: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.dynamic or self.shape != shape:
Running MIL default pipeline:  11%|█         | 10/95 [00:00<00:01, 44.22 passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1444', of the source model, has been renamed to 'var_1444' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL default pipeline:  65%|██████▌   | 62/95 [00:02<00:03,  9.47 passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur

CoreML: export success ✅ 22.5s, saved as '/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.mlpackage' (18.3 MB)

Export complete (23.5s)
Results saved to /content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights
Predict:         yolo predict task=detect model=/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.mlpackage imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/ParkFlow_AI/training_runs/parkflow_final_vehicle_model/weights/best.mlpackage imgsz=640 data=/content/raw_data/Vehicle_Detection_Image_Dataset/data.yaml  
Visualize:       https://netron.app
All exports complete. Files are located in the same folder as your best.pt
